# 面试问题：你会怎样设计 LLM 推理的 continuous batching 调度器？

**一句话回答**：我不会从“把请求凑成一个 batch”开始，而会先定义 TTFT、TPOT、deadline、公平性和显存上限；调度器把 prefill 与 decode 当成不同成本的工作，逐轮接纳/移除序列，用 token budget、chunked prefill、租户配额和 admission control 控制尾延迟，并用可重放 trace 验证吞吐与 SLO。

本 Notebook 不调用推理框架，而用 Python/NumPy 写一个离散事件调度器，明确展示请求状态机、静态批处理的队头阻塞、continuous batching、deadline、取消、背压和发布合同。

In [ ]:
from dataclasses import dataclass, field  # 导入本单元所需的依赖。
from collections import Counter, defaultdict, deque  # 导入本单元所需的依赖。
import hashlib, json, math  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

SEED79=7901  # 计算并保存当前步骤的中间状态。
rng79=np.random.default_rng(SEED79)  # 计算并保存当前步骤的中间状态。
assert np.__version__ and SEED79==7901  # 用受控断言验证关键不变量。
assert math.isclose(3/2,1.5)  # 用受控断言验证关键不变量。
assert hashlib.sha256(b"scheduler").hexdigest()!=hashlib.sha256(b"cache").hexdigest()  # 用受控断言验证关键不变量。

## 1. 先把问题变成可验证合同

请求至少包含到达时间、prompt token 数、最大生成长度、deadline、租户和优先级。输出不能只有文本，还要记录 admission、首 token、完成/拒绝/取消原因。TTFT 是 `first_token-arrival`，TPOT 是 decode 阶段 token 间隔，吞吐是完成 token/时间；三者不能互相替代。

这里将请求状态限定为 `queued -> prefill -> decode -> done`，拒绝和取消是终态。任何状态倒退、重复完成或超过槽位/预算都应由断言发现。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Request79:  # 定义承载本节状态与行为的数据结构。
    request_id:str; tenant:str; arrival:int; prompt_tokens:int; max_new_tokens:int; deadline:int; priority:int=0  # 计算并保存当前步骤的中间状态。
    def __post_init__(self):  # 定义本节可复用的核心函数。
        if not self.request_id or self.arrival<0 or self.prompt_tokens<1 or self.max_new_tokens<1: raise ValueError("request_contract")  # 按当前条件选择后续控制路径。
        if self.deadline<=self.arrival or self.priority not in (0,1): raise ValueError("request_contract")  # 按当前条件选择后续控制路径。

@dataclass  # 为下方定义附加声明式配置。
class State79:  # 定义承载本节状态与行为的数据结构。
    req:Request79; phase:str="queued"; prefill_left:int=0; generated:int=0; admitted_at:int|None=None; first_token_at:int|None=None; finished_at:int|None=None; reason:str|None=None  # 计算并保存当前步骤的中间状态。
    def __post_init__(self): self.prefill_left=self.req.prompt_tokens  # 定义本节可复用的核心函数。

probe79=Request79("r0","tenant-a",0,8,3,20,1); state79=State79(probe79)  # 计算并保存当前步骤的中间状态。
assert state79.phase=="queued" and state79.prefill_left==8 and state79.generated==0  # 用受控断言验证关键不变量。
assert probe79.deadline-probe79.arrival==20  # 用受控断言验证关键不变量。
try: Request79("","x",0,1,1,2); raise AssertionError("invalid request accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="request_contract"  # 捕获预期异常并验证失败分支。

## 2. 构造能暴露队头阻塞的 trace

工作负载混合长 prompt/长生成与短交互请求。静态 batching 只有整批结束才接纳下一批，短请求会被前一批最长序列拖住；continuous batching 在槽位释放后立即补入新请求。受控 trace 不是性能 benchmark，而是算法 oracle。

到达过程必须固定 seed 并保存，否则两个调度策略面对不同负载，比较没有意义。

In [ ]:
requests79=[]  # 计算并保存当前步骤的中间状态。
for i in range(72):  # 遍历输入元素以累积或检查结果。
    arrival=i//3  # 计算并保存当前步骤的中间状态。
    prompt=int(rng79.choice([4,8,12,28],p=[.35,.35,.2,.1]))  # 计算并保存当前步骤的中间状态。
    new=int(rng79.choice([2,4,8,20],p=[.4,.35,.2,.05]))  # 计算并保存当前步骤的中间状态。
    tenant=["a","b","c"][i%3]  # 计算并保存当前步骤的中间状态。
    requests79.append(Request79(f"r{i:02d}",tenant,arrival,prompt,new,arrival+300,1 if i%17==0 else 0))  # 计算并保存当前步骤的中间状态。
assert len(requests79)==72 and len({r.request_id for r in requests79})==72  # 用受控断言验证关键不变量。
assert sum(r.prompt_tokens>=28 for r in requests79)>0 and sum(r.max_new_tokens<=4 for r in requests79)>30  # 用受控断言验证关键不变量。
assert requests79==sorted(requests79,key=lambda r:(r.arrival,r.request_id))  # 用受控断言验证关键不变量。

## 3. 手写逐轮调度器

每个 tick 有固定 token budget。prefill 可切块，避免一个超长 prompt 独占整轮；decode 每个活跃序列一次只生成一个 token，近似 GPU 上同步 decode iteration。`mode=static` 只在 active 为空时接纳，`continuous` 每轮补空槽。

为防止 priority 请求无限挤压普通租户，admission key 先比较“是否临近 deadline”，再比较优先级和到达顺序；每租户设置 active cap。真实系统还要用 KV block 数而不是请求数做显存 admission。

In [ ]:
class Scheduler79:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,mode,max_slots=8,token_budget=24,prefill_chunk=8,tenant_cap=4):  # 定义本节可复用的核心函数。
        if mode not in {"static","continuous"} or min(max_slots,token_budget,prefill_chunk,tenant_cap)<1: raise ValueError("scheduler_contract")  # 按当前条件选择后续控制路径。
        self.mode=mode; self.max_slots=max_slots; self.token_budget=token_budget; self.prefill_chunk=prefill_chunk; self.tenant_cap=tenant_cap  # 计算并保存当前步骤的中间状态。
    def run(self,requests,max_time=500):  # 定义本节可复用的核心函数。
        states={r.request_id:State79(r) for r in requests}; pending=[]; active=[]; events=[]; completed_tokens=0  # 计算并保存当前步骤的中间状态。
        for now in range(max_time):  # 遍历输入元素以累积或检查结果。
            for r in requests:  # 遍历输入元素以累积或检查结果。
                if r.arrival==now: pending.append(states[r.request_id]); events.append((now,r.request_id,"arrive"))  # 按当前条件选择后续控制路径。
            for s in list(pending):  # 遍历输入元素以累积或检查结果。
                if now>=s.req.deadline: s.phase="rejected"; s.reason="deadline_before_admission"; s.finished_at=now; pending.remove(s)  # 按当前条件选择后续控制路径。
            can_admit=self.mode=="continuous" or not active  # 计算并保存当前步骤的中间状态。
            if can_admit:  # 按当前条件选择后续控制路径。
                tenant_counts=Counter(s.req.tenant for s in active)  # 计算并保存当前步骤的中间状态。
                pending.sort(key=lambda s:(now+10<s.req.deadline,-s.req.priority,s.req.arrival,s.req.request_id))  # 计算并保存当前步骤的中间状态。
                for s in list(pending):  # 遍历输入元素以累积或检查结果。
                    if len(active)>=self.max_slots: break  # 按当前条件选择后续控制路径。
                    if tenant_counts[s.req.tenant]>=self.tenant_cap: continue  # 按当前条件选择后续控制路径。
                    s.phase="prefill"; s.admitted_at=now; active.append(s); pending.remove(s); tenant_counts[s.req.tenant]+=1; events.append((now,s.req.request_id,"admit"))  # 计算并保存当前步骤的中间状态。
            budget=self.token_budget  # 计算并保存当前步骤的中间状态。
            for s in list(active):  # 遍历输入元素以累积或检查结果。
                if budget<=0: break  # 按当前条件选择后续控制路径。
                if s.phase=="prefill":  # 按当前条件选择后续控制路径。
                    used=min(s.prefill_left,self.prefill_chunk,budget); s.prefill_left-=used; budget-=used  # 计算并保存当前步骤的中间状态。
                    if s.prefill_left==0: s.phase="decode"  # 按当前条件选择后续控制路径。
            for s in list(active):  # 遍历输入元素以累积或检查结果。
                if budget<=0: break  # 按当前条件选择后续控制路径。
                if s.phase=="decode":  # 按当前条件选择后续控制路径。
                    s.generated+=1; completed_tokens+=1; budget-=1  # 计算并保存当前步骤的中间状态。
                    if s.first_token_at is None: s.first_token_at=now; events.append((now,s.req.request_id,"first_token"))  # 按当前条件选择后续控制路径。
                    if s.generated==s.req.max_new_tokens:  # 按当前条件选择后续控制路径。
                        s.phase="done"; s.finished_at=now+1; active.remove(s); events.append((now+1,s.req.request_id,"done"))  # 计算并保存当前步骤的中间状态。
            assert len(active)<=self.max_slots and budget>=0  # 用受控断言验证关键不变量。
            if not pending and not active and now>=max(r.arrival for r in requests): return states,events,now+1,completed_tokens  # 按当前条件选择后续控制路径。
        raise RuntimeError("simulation_timeout")  # 遇到非法合同立即显式失败。

static79=Scheduler79("static"); continuous79=Scheduler79("continuous")  # 计算并保存当前步骤的中间状态。
s_states79,s_events79,s_time79,s_tokens79=static79.run(requests79)  # 计算并保存当前步骤的中间状态。
c_states79,c_events79,c_time79,c_tokens79=continuous79.run(requests79)  # 计算并保存当前步骤的中间状态。
assert s_tokens79==c_tokens79==sum(r.max_new_tokens for r in requests79)  # 用受控断言验证关键不变量。
assert all(s.phase=="done" for s in c_states79.values()) and c_time79<s_time79  # 用受控断言验证关键不变量。
assert len([e for e in c_events79 if e[2]=="done"])==72  # 用受控断言验证关键不变量。

## 4. 计算 TTFT、完成延迟与公平性

尾延迟必须按请求统计，吞吐按 token 统计。只报均值会掩盖长 prompt 和低优先级租户。这里还计算租户平均 slowdown：`latency / (prompt+output)`；若某租户显著恶化，需要调低 cap、引入 deficit round robin 或保留容量。

static 与 continuous 的输出 token 相同，因此延迟改善不是少做工作换来的。

In [ ]:
def metrics79(states,total_time):  # 定义本节可复用的核心函数。
    done=[s for s in states.values() if s.phase=="done"]  # 计算并保存当前步骤的中间状态。
    ttft=np.array([s.first_token_at-s.req.arrival for s in done],float); latency=np.array([s.finished_at-s.req.arrival for s in done],float)  # 计算并保存当前步骤的中间状态。
    tenant=defaultdict(list)  # 计算并保存当前步骤的中间状态。
    for s in done: tenant[s.req.tenant].append((s.finished_at-s.req.arrival)/(s.req.prompt_tokens+s.req.max_new_tokens))  # 遍历输入元素以累积或检查结果。
    return {"count":len(done),"ttft_p50":float(np.quantile(ttft,.5)),"ttft_p95":float(np.quantile(ttft,.95)),"latency_p95":float(np.quantile(latency,.95)),"throughput":sum(s.generated for s in done)/total_time,"tenant_slowdown":{k:float(np.mean(v)) for k,v in tenant.items()}}  # 返回当前分支计算出的结果。
sm79=metrics79(s_states79,s_time79); cm79=metrics79(c_states79,c_time79)  # 计算并保存当前步骤的中间状态。
assert cm79["count"]==sm79["count"]==72 and cm79["throughput"]>sm79["throughput"]  # 用受控断言验证关键不变量。
assert cm79["ttft_p95"]<sm79["ttft_p95"] and cm79["latency_p95"]<=sm79["latency_p95"]  # 用受控断言验证关键不变量。
assert max(cm79["tenant_slowdown"].values())/min(cm79["tenant_slowdown"].values())<1.5  # 用受控断言验证关键不变量。

## 5. 槽位不是显存：用 KV block 做容量估算

两个请求都占一个 active slot，但 4K prompt 与 32 token prompt 的 KV 显存完全不同。分页 KV 中可将 `prompt + 最大输出` 向上取整为 block 数，admission 同时约束 active sequence 和总 block。估计过于保守会损失吞吐，过于乐观会在 decode 中途 OOM。

真实系统还应区分已分配与可增长 block，并为 beam/speculative token、CUDA graph 和碎片预留 headroom。

In [ ]:
def kv_blocks79(req,block_tokens=16):  # 定义本节可复用的核心函数。
    if block_tokens<1: raise ValueError("block_contract")  # 按当前条件选择后续控制路径。
    return math.ceil((req.prompt_tokens+req.max_new_tokens)/block_tokens)  # 返回当前分支计算出的结果。
block_usage79={r.request_id:kv_blocks79(r) for r in requests79}  # 计算并保存当前步骤的中间状态。
assert all(v>=1 for v in block_usage79.values())  # 用受控断言验证关键不变量。
assert kv_blocks79(Request79("short","a",0,4,2,10))==1  # 用受控断言验证关键不变量。
assert kv_blocks79(Request79("long","a",0,28,20,60))==3  # 用受控断言验证关键不变量。

## 6. 取消、背压与 admission control

已断开的请求应尽快释放 KV；但取消事件必须带 request generation，防止旧取消误杀复用 ID 的新请求。队列过长时，宁可在 admission 前明确拒绝，也不要让所有请求超时。估算工作量可用 `prompt + expected_output`，不能只看请求数。

下例实现纯函数 admission：超过队列 token 上限、deadline 已无可行余量时返回稳定原因码，便于客户端重试和监控聚合。

In [ ]:
def admit79(req,now,queued_work,max_queue_tokens=300,min_service_rate=8):  # 定义本节可复用的核心函数。
    if now>=req.deadline: return False,"deadline_expired"  # 按当前条件选择后续控制路径。
    work=req.prompt_tokens+req.max_new_tokens  # 计算并保存当前步骤的中间状态。
    if queued_work+work>max_queue_tokens: return False,"queue_token_limit"  # 按当前条件选择后续控制路径。
    estimated=math.ceil((queued_work+work)/min_service_rate)  # 计算并保存当前步骤的中间状态。
    if now+estimated>req.deadline: return False,"deadline_infeasible"  # 按当前条件选择后续控制路径。
    return True,"accepted"  # 返回当前分支计算出的结果。
assert admit79(Request79("x","a",0,8,4,20),0,20)==(True,"accepted")  # 用受控断言验证关键不变量。
assert admit79(Request79("y","a",0,8,4,20),0,295)[1]=="queue_token_limit"  # 用受控断言验证关键不变量。
assert admit79(Request79("z","a",0,8,4,2),0,20)[1]=="deadline_infeasible"  # 用受控断言验证关键不变量。

## 7. Trace 不变量与故障定位

调度器的回归集不只检查聚合指标，还要验证每个请求事件单调：到达先于接纳，接纳先于首 token，首 token 先于完成；每个请求最多一个终态。线上出现 TTFT 抖动时，可按阶段区分 queue、prefill 和 decode，而不是笼统归因“GPU 慢”。

Trace 中记录长度 bucket、调度版本和原因码即可，prompt 原文通常不应进入调度日志。

In [ ]:
def validate_trace79(events):  # 定义本节可复用的核心函数。
    by_req=defaultdict(list)  # 计算并保存当前步骤的中间状态。
    for t,rid,event in events: by_req[rid].append((t,event))  # 遍历输入元素以累积或检查结果。
    for rid,seq in by_req.items():  # 遍历输入元素以累积或检查结果。
        times=[t for t,_ in seq]; names=[e for _,e in seq]  # 计算并保存当前步骤的中间状态。
        if times!=sorted(times) or names[0]!="arrive" or names.count("done")>1: return False  # 按当前条件选择后续控制路径。
        if "done" in names and not names.index("admit")<names.index("first_token")<names.index("done"): return False  # 按当前条件选择后续控制路径。
    return True  # 返回当前分支计算出的结果。
assert validate_trace79(c_events79) and validate_trace79(s_events79)  # 用受控断言验证关键不变量。
assert all(sum(1 for e in c_events79 if e[1]==r.request_id and e[2]=="done")==1 for r in requests79)  # 用受控断言验证关键不变量。
assert all(c_states79[r.request_id].first_token_at<c_states79[r.request_id].finished_at for r in requests79)  # 用受控断言验证关键不变量。

## 8. 发布、回放与回滚

调度参数会改变延迟和公平性，属于可版本化制品：包括 max slots、token budget、prefill chunk、租户 cap、模型/KV dtype 和代码版本。上线前用固定 trace 回放，比较 SLO 与吞吐；线上用 shadow scheduler 只做决策不执行，确认新旧差异。

制品摘要必须覆盖参数，加载时重算；只校验文件名无法防止静默配置漂移。

In [ ]:
manifest79={"schema":1,"algorithm":"continuous_batching","max_slots":8,"token_budget":24,"prefill_chunk":8,"tenant_cap":4,"seed":SEED79}  # 计算并保存当前步骤的中间状态。
payload79=json.dumps(manifest79,sort_keys=True,separators=(",",":")); digest79=hashlib.sha256(payload79.encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
artifact79={"manifest":manifest79,"sha256":digest79,"replay":{"requests":len(requests79),"ttft_p95":cm79["ttft_p95"],"throughput":cm79["throughput"]}}  # 计算并保存当前步骤的中间状态。
assert hashlib.sha256(json.dumps(artifact79["manifest"],sort_keys=True,separators=(",",":")).encode()).hexdigest()==artifact79["sha256"]  # 用受控断言验证关键不变量。
forged79=json.loads(json.dumps(artifact79)); forged79["manifest"]["tenant_cap"]=99  # 计算并保存当前步骤的中间状态。
assert hashlib.sha256(json.dumps(forged79["manifest"],sort_keys=True,separators=(",",":")).encode()).hexdigest()!=forged79["sha256"]  # 用受控断言验证关键不变量。
assert artifact79["replay"]["requests"]==72 and artifact79["manifest"]["algorithm"]=="continuous_batching"  # 用受控断言验证关键不变量。

## 9. 面试收束与进一步追问

完整回答应形成闭环：SLO/请求合同 → prefill/decode 状态机 → token/KV admission → continuous batching → deadline/公平性 → trace 回放 → 灰度与回滚。不要说“动态 batching 提高吞吐”就结束，还要解释长 prompt、取消、租户隔离和尾延迟。

进一步追问：Paged KV allocator 如何把槽位约束换成 block 约束？prefill/decode 是否拆到不同 GPU？如何处理 speculative decoding 的可变接受长度？如何做 prefix cache 感知调度？

研究入口：[Orca/iteration-level scheduling](https://www.usenix.org/conference/osdi22/presentation/yu)、[vLLM/PagedAttention](https://arxiv.org/abs/2309.06180)、[DistServe](https://arxiv.org/abs/2401.09670)。